In [ ]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j

In [ ]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    #fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/aura_api_log.log",'a',encoding='utf-8')
    #fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    #fh.setFormatter(formatter)
    logger.addHandler(ch)
    #logger.addHandler(fh)
    return logger
logger=logging.getLogger()

In [ ]:
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"


neo=Neo4j(uri,user,password)
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

In [ ]:
uri2='neo4j+ssc://bd2c184f.databases.neo4j.io'
user2='neo4j'
password2='OkvWgegnLgUSuAPl-Zcu8wkcXCX-GxNRC2zydwpbwrE'
neo2=Neo4j(uri2,user2,password2)

In [ ]:
mysql.close()
mysql=MYSQL(host,user,password,database)

In [ ]:
def neo2mysql():
    cypher='match (n) return id(n),labels(n),n.name,n.code'
    result=neo.read_query(cypher)
    val=[]
    for r in result:
        val.append((str(r.values()[1])[1:-1],r.values()[2],r.values()[3],r.values()[0]))
    val1=[]
    for v in val:
        val1.append(v+(v[0],v[1],v[2]))

    name_val=[]
    for v in val:
        name_val.append((v[3],'name',v[1],v[1]))
    code_val=[]
    for v in val:
        code_val.append((v[3],'code',v[2],v[2]))
    sql='insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1'
    mysql.write_many_query(sql,val1)

    sql2='insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s'
    mysql.write_many_query(sql2,name_val)
    mysql.write_many_query(sql2,code_val)

    mysql.write_query('SET FOREIGN_KEY_CHECKS = 0')
    mysql.write_query('delete from edges')
    mysql.write_query('SET FOREIGN_KEY_CHECKS = 1')
    cypher='match()-[r]->() return id(startNode(r)),id(endNode(r)),type(r),id(r)  '
    result=neo.read_query(cypher)
    val=[]
    for r in result:
        val.append((str(r.values()[0]),r.values()[1],r.values()[2],r.values()[3],r.values()[2],r.values()[3]))
    sql='insert into edges(start,end,relationship,id,using_flag) value(%s,%s,%s,%s,1) on duplicate key update relationship=%s,id=%s,using_flag=1'
    mysql.write_many_query(sql,val)



In [ ]:
neo2mysql()

In [ ]:
#result[0]
for r in result:
    id=r[0]
    name=r[2]
    code=r[3]
    if code:
        if name:
            cypher=('match (n)  where id(n)=%d set n.name="%s",n.code="%s"'%(id,name,code))
        else:
            cypher=('match (n) where id(n)=%d set n.code="%s"'%(id,code))
    else:
        if name:
            cypher=('match (n)  where id(n)=%d set n.name="%s"'%(id,name))
        else:
            pass
    neo2.simple_query(cypher)

In [ ]:
val=[]
for r in result:
    val.append((str(r.values()[1])[1:-1],r.values()[2],r.values()[3],r.values()[0]))
val

In [ ]:
val1=[]
for v in val:
    val1.append(v+(v[0],v[1],v[2]))
val1
name_val=[]
for v in val:
    name_val.append((v[3],'name',v[1],v[1]))
code_val=[]
for v in val:
    code_val.append((v[3],'code',v[2],v[2]))
code_val

In [ ]:
len(val1)

In [ ]:
sql='insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1'
mysql.write_many_query(sql,val1)

sql2='insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s'
mysql.write_many_query(sql2,name_val)


In [ ]:

mysql.write_many_query(sql2,code_val)

In [ ]:
cypher='match()-[r]->() return id(startNode(r)),id(endNode(r)),type(r),id(r)  '
result=neo.read_query(cypher)
result

In [ ]:
len(result)

In [ ]:
val=[]
for r in result:
    val.append((str(r.values()[0]),r.values()[1],r.values()[2],r.values()[3],r.values()[2],r.values()[3]))
sql='insert into edges(start,end,relationship,id,using_flag) value(%s,%s,%s,%s,1) on duplicate key update relationship=%s,id=%s,using_flag=1'
mysql.write_many_query(sql,val)

In [ ]:

sql='update nodes set labels=%s ,name=%s,using_flag=1 where id =%s'
r=mysql.write_many_query(sql,val)
r

In [ ]:
def refresh_flag():
    
    cypher='match (n) return id(n)'
    r=neo.simple_query(cypher)
    using=[]
    for record in r:
        using.append((record.value(),))
    sql0='update nodes set using_flag=0'
    sql='update nodes set using_flag=1 where id=%s'
    mysql.write_query(sql0)
    n=mysql.write_many_query(sql,using)
    print(n)
    
    cypher='match ()-[r]->() return id(r)'
    r=neo.simple_query(cypher)
    using=[]
    for record in r:
        using.append((record.value(),))
    sql0='update edges set using_flag=0'
    sql='update edges set using_flag=1 where id=%s'
    mysql.write_query(sql0)
    e=mysql.write_many_query(sql,using)
    print(e)